In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🏀 Multi-Object Detection & Persistent ID Tracking\n",
    "### Sports Footage — YOLOv8 + ByteTrack + PersistentIDManager\n",
    "---\n",
    "**Pipeline Overview:**\n",
    "1. Download a public sports video\n",
    "2. Detect all players using YOLOv8m\n",
    "3. Assign stable IDs using ByteTrack + our custom PersistentIDManager\n",
    "4. Visualize bounding boxes, IDs, and motion trails\n",
    "5. Generate heatmap and count-over-time analytics"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## ⚙️ Step 0 — Install Dependencies"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Run this once\n",
    "!pip install ultralytics opencv-python supervision matplotlib pandas yt-dlp Pillow -q\n",
    "# PyTorch with CUDA (Windows)\n",
    "!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 📥 Step 1 — Download Public Sports Video"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import subprocess, os\n",
    "\n",
    "VIDEO_URL    = 'https://www.youtube.com/watch?v=eAONrDFJaKU'\n",
    "INPUT_VIDEO  = '../input_video.mp4'\n",
    "OUTPUT_VIDEO = '../output/tracked.mp4'\n",
    "\n",
    "os.makedirs('../output', exist_ok=True)\n",
    "os.makedirs('../output/screenshots', exist_ok=True)\n",
    "\n",
    "if not os.path.exists(INPUT_VIDEO):\n",
    "    print('Downloading video...')\n",
    "    subprocess.run([\n",
    "        'yt-dlp', '-f', 'best[ext=mp4][height<=720]',\n",
    "        '-o', INPUT_VIDEO, VIDEO_URL\n",
    "    ])\n",
    "    print('✅ Download complete!')\n",
    "else:\n",
    "    print('✅ Video already exists, skipping download.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 🔍 Step 2 — Inspect Video Info"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import cv2, sys\n",
    "sys.path.append('..')\n",
    "\n",
    "cap = cv2.VideoCapture(INPUT_VIDEO)\n",
    "info = {\n",
    "    'Width'       : int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),\n",
    "    'Height'      : int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),\n",
    "    'FPS'         : round(cap.get(cv2.CAP_PROP_FPS), 2),\n",
    "    'Total Frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),\n",
    "}\n",
    "cap.release()\n",
    "\n",
    "for k, v in info.items():\n",
    "    print(f'  {k:<15}: {v}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 🚀 Step 3 — Run Full Tracking Pipeline"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import cv2, time, sys, pandas as pd\n",
    "sys.path.append('..')\n",
    "\n",
    "from tracker    import SportsTracker\n",
    "from visualizer import Visualizer\n",
    "from id_manager import PersistentIDManager\n",
    "from utils      import get_video_info, create_video_writer\n",
    "\n",
    "# ── Config ──────────────────────────────────────────────────\n",
    "CONF_THRESH  = 0.35\n",
    "DEVICE       = 'cuda'   # change to 'cpu' if no GPU\n",
    "GRACE        = 45\n",
    "IOU_THRESH   = 0.25\n",
    "HIST_THRESH  = 0.45\n",
    "MAX_FRAMES   = 500      # set -1 for full video\n",
    "\n",
    "# ── Init ────────────────────────────────────────────────────\n",
    "info       = get_video_info(INPUT_VIDEO)\n",
    "W, H, FPS  = info['width'], info['height'], info['fps']\n",
    "\n",
    "tracker    = SportsTracker(conf_threshold=CONF_THRESH, device=DEVICE)\n",
    "visualizer = Visualizer(draw_trails=True, trail_length=40)\n",
    "id_manager = PersistentIDManager(GRACE, IOU_THRESH, HIST_THRESH)\n",
    "\n",
    "cap        = cv2.VideoCapture(INPUT_VIDEO)\n",
    "writer     = create_video_writer(OUTPUT_VIDEO, FPS, W, H)\n",
    "\n",
    "frame_log, count_log = [], []\n",
    "frame_idx = total_proc = 0\n",
    "start = time.time()\n",
    "\n",
    "print(f'Processing up to {MAX_FRAMES} frames...\\n')\n",
    "\n",
    "while cap.isOpened():\n",
    "    ret, frame = cap.read()\n",
    "    if not ret: break\n",
    "    if MAX_FRAMES > 0 and frame_idx >= MAX_FRAMES: break\n",
    "\n",
    "    detections = tracker.track(frame)\n",
    "    detections = id_manager.update(detections, frame)\n",
    "\n",
    "    for det in detections:\n",
    "        frame = visualizer.draw(frame, det['final_id'],\n",
    "                                det['x1'], det['y1'], det['x2'], det['y2'])\n",
    "        frame_log.append({'frame': frame_idx, 'final_id': det['final_id'],\n",
    "                          'conf': round(det['conf'],3)})\n",
    "\n",
    "    active = len(detections)\n",
    "    frame  = visualizer.draw_count(frame, active)\n",
    "    count_log.append({'frame': frame_idx, 'count': active})\n",
    "\n",
    "    if frame_idx % 100 == 0:\n",
    "        cv2.imwrite(f'../output/screenshots/frame_{frame_idx:05d}.jpg', frame)\n",
    "        print(f'  Frame {frame_idx:>5} | Active: {active:>3} | '\n",
    "              f'Elapsed: {time.time()-start:.1f}s')\n",
    "\n",
    "    writer.write(frame)\n",
    "    frame_idx  += 1\n",
    "    total_proc += 1\n",
    "\n",
    "cap.release()\n",
    "writer.release()\n",
    "\n",
    "df  = pd.DataFrame(frame_log)\n",
    "cdf = pd.DataFrame(count_log)\n",
    "df.to_csv('../output/tracking_log.csv', index=False)\n",
    "cdf.to_csv('../output/count_over_time.csv', index=False)\n",
    "\n",
    "s = id_manager.summary()\n",
    "print(f'\\n✅ Done in {time.time()-start:.1f}s')\n",
    "print(f'   Unique stable IDs : {s[\"total_unique_ids\"]}')\n",
    "print(f'   Total detections  : {len(df)}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 🖼️ Step 4 — Preview Sample Frames"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os, glob\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.image as mpimg\n",
    "\n",
    "shots = sorted(glob.glob('../output/screenshots/*.jpg'))[:6]\n",
    "fig, axes = plt.subplots(2, 3, figsize=(18, 9))\n",
    "for ax, path in zip(axes.flatten(), shots):\n",
    "    ax.imshow(mpimg.imread(path))\n",
    "    ax.set_title(os.path.basename(path), fontsize=9)\n",
    "    ax.axis('off')\n",
    "plt.suptitle('Sample Tracked Frames', fontsize=14, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../output/sample_frames.png', dpi=150)\n",
    "plt.show()\n",
    "print('✅ Saved → output/sample_frames.png')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 🔥 Step 5 — Movement Heatmap"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd, numpy as np, cv2\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "df  = pd.read_csv('../output/tracking_log.csv')\n",
    "cap = cv2.VideoCapture(INPUT_VIDEO)\n",
    "W2  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n",
    "H2  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n",
    "cap.release()\n",
    "\n",
    "# Rebuild cx/cy from log\n",
    "full = pd.read_csv('../output/tracking_log.csv')\n",
    "\n",
    "heatmap = np.zeros((H2, W2), dtype=np.float32)\n",
    "\n",
    "# Re-read with x1/y1/x2/y2 for centroid\n",
    "tlog = pd.read_csv('../output/tracking_log.csv')\n",
    "if 'x1' in tlog.columns:\n",
    "    tlog['cx'] = ((tlog['x1'] + tlog['x2']) / 2).astype(int)\n",
    "    tlog['cy'] = ((tlog['y1'] + tlog['y2']) / 2).astype(int)\n",
    "    for _, row in tlog.iterrows():\n",
    "        cx, cy = int(row['cx']), int(row['cy'])\n",
    "        if 0 <= cx < W2 and 0 <= cy < H2:\n",
    "            cv2.circle(heatmap, (cx, cy), 18, 1, -1)\n",
    "\n",
    "heatmap = cv2.GaussianBlur(heatmap, (61, 61), 0)\n",
    "norm    = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)\n",
    "colored = cv2.applyColorMap(norm, cv2.COLORMAP_JET)\n",
    "cv2.imwrite('../output/heatmap.png', colored)\n",
    "\n",
    "plt.figure(figsize=(13, 7))\n",
    "plt.imshow(cv2.cvtColor(colored, cv2.COLOR_BGR2RGB))\n",
    "plt.title('Movement Heatmap — All Tracked Subjects', fontsize=13)\n",
    "plt.axis('off')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../output/heatmap_display.png', dpi=150)\n",
    "plt.show()\n",
    "print('✅ Heatmap saved → output/heatmap.png')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 📈 Step 6 — Active Subject Count Over Time"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "cdf = pd.read_csv('../output/count_over_time.csv')\n",
    "\n",
    "plt.figure(figsize=(14, 5))\n",
    "plt.plot(cdf['frame'], cdf['count'], color='royalblue', linewidth=1.5)\n",
    "plt.fill_between(cdf['frame'], cdf['count'], alpha=0.2, color='royalblue')\n",
    "plt.axhline(cdf['count'].mean(), color='red', linestyle='--',\n",
    "            linewidth=1, label=f\"Avg: {cdf['count'].mean():.1f}\")\n",
    "plt.xlabel('Frame Number', fontsize=11)\n",
    "plt.ylabel('Active Subjects', fontsize=11)\n",
    "plt.title('Active Tracked Subjects Over Time', fontsize=13)\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../output/count_over_time.png', dpi=150)\n",
    "plt.show()\n",
    "print('✅ Count plot saved → output/count_over_time.png')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 📊 Step 7 — Tracking Statistics"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "\n",
    "df  = pd.read_csv('../output/tracking_log.csv')\n",
    "cdf = pd.read_csv('../output/count_over_time.csv')\n",
    "\n",
    "print('='*45)\n",
    "print('  TRACKING STATISTICS')\n",
    "print('='*45)\n",
    "print(f'  Unique stable IDs     : {df[\"final_id\"].nunique()}')\n",
    "print(f'  Total detections      : {len(df)}')\n",
    "print(f'  Avg detections/frame  : {len(df)/len(cdf):.2f}')\n",
    "print(f'  Max active at once    : {cdf[\"count\"].max()}')\n",
    "print(f'  Min active at once    : {cdf[\"count\"].min()}')\n",
    "print(f'  Avg active per frame  : {cdf[\"count\"].mean():.1f}')\n",
    "print('='*45)\n",
    "\n",
    "# Per-ID detection count\n",
    "per_id = df.groupby('final_id').size().reset_index(name='detections')\n",
    "per_id = per_id.sort_values('detections', ascending=False)\n",
    "print('\\nTop 10 most tracked subjects:')\n",
    "print(per_id.head(10).to_string(index=False))"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}